In [ ]:
import pandas as pd
import numpy as np
rng = np.random.default_rng(42)

# Load data
phys = pd.read_csv("physiological_cycles.csv")
slp  = pd.read_csv("sleeps.csv")
wkt  = pd.read_csv("workouts.csv")

# Convert timestamps to datetimes and extract a daily key
# For physiological and sleep data, use "Cycle start time" as the reference
phys["Cycle start time"] = pd.to_datetime(phys["Cycle start time"], errors="coerce")
slp["Cycle start time"]  = pd.to_datetime(slp["Cycle start time"],  errors="coerce")

phys["day"] = phys["Cycle start time"].dt.date
slp["day"]  = slp["Cycle start time"].dt.date

# For workouts, use "Workout start time" as the reference
wkt["Workout start time"] = pd.to_datetime(wkt["Workout start time"], errors="coerce")
wkt["day"] = wkt["Workout start time"].dt.date

# Create a sorted list
# Which is going to be the index for all parameters
unique_days = sorted(set(phys["day"]) | set(slp["day"]) | set(wkt["day"]))

# steps: daily step count.
# autogenerated
steps = rng.normal(10000, 2500, len(unique_days)).clip(3000, 20000)

# tib: "time in bed" per day in HOURS.
# "In bed duration (min)" summed per day.
tib = (
    slp.groupby("day")["In bed duration (min)"]
       .sum()
       .reindex(unique_days, fill_value=0)
       .to_numpy() / 60.0
)

# ac: "active calories burned" per day in CALORIES.
# "Energy burned (kcal)" summed per day.
ac = (
    phys.groupby("day")["Energy burned (cal)"]
        .sum()
        .reindex(unique_days, fill_value=0)
        .to_numpy() / 1000.0
)

# exm: "exercise minutes" per day in MINUTES.
# "Duration (min)" for exercise summed per day.
exm = (
    wkt.groupby("day")["Duration (min)"]
       .sum()
       .reindex(unique_days, fill_value=0)
       .to_numpy()
)

# nsed: "non-sedentary hours" per day in HOURS.
# total hours in a day minus time spent in be
nsed = 24 - tib

# dlh: "daylight hours" per day in HOURS.
# autogenerated
# correlate slightly with non-sedentary hours (nsed)
dlh = (rng.normal(1.5, 0.7, len(unique_days)) + (nsed - 14) * 0.1).clip(0.5, 4.0)

# dist: "distance walked" per day in METERS.
# autogenerated
# linked to steps (avg step ≈ 0.78m)
dist = (steps * rng.normal(0.75, 0.05, len(unique_days))).clip(1000, 20000)

# flt: "flights of stairs climbed" per day (COUNT).
# autogenerated
# loosely based on intensity of the workout
flt = rng.normal(8, 6, len(unique_days)).clip(0, 30)

# rhr: "average resting heart rate" per day in BPM.
# "Resting heart rate (bpm)".
rhr = (
    phys.groupby("day")["Resting heart rate (bpm)"]
        .mean()
        .reindex(unique_days, fill_value=0)
        .to_numpy()
)

steps = steps.astype(int)
dist = dist.astype(int)
flt = np.round(flt).astype(int)

# Stack everything into X_raw in order:
X_raw = np.vstack([steps, tib, ac, exm, nsed, dlh, dist, flt, rhr]).T
np.set_printoptions(suppress=True)

print("X_raw shape:", X_raw.shape)
print("First few rows of X_raw:\n", X_raw[:5])

X_raw shape: (618, 9)
First few rows of X_raw:
 [[  0.           8.3          3.584        0.          15.7
    0.           0.           0.          54.        ]
 [  0.           0.           0.          92.          24.
    0.           0.           0.           0.        ]
 [  0.           9.08333333   2.394      139.          14.91666667
    0.           0.           0.          53.        ]
 [  0.           8.95         2.443       71.          15.05
    0.           0.           0.          55.        ]
 [  0.           8.96666667   2.304        0.          15.03333333
    0.           0.           0.          53.        ]]
